# Probabilistic N-gram Language Model for Darija

This notebook walks through the full pipeline of building and evaluating an interpolated trigram language model trained on a Darija (Moroccan Arabic) corpus.

**Approach:** Linear interpolation combining unigram, bigram, and trigram probabilities.
Lambda weights are estimated from training data via **deleted interpolation** (Jelinek & Mercer, 1980).

$$P(w \mid w_1, w_2) = \lambda_1 P(w) + \lambda_2 P(w \mid w_2) + \lambda_3 P(w \mid w_1, w_2)$$

## 1. Setup

In [1]:
import random
from data_loader import load_corpus
from preprocessor import preprocess
from ngram_model import NgramModel
from evaluator import perplexity
from generator import generate

SEED = 42
random.seed(SEED)

## 2. Load Corpus

Training data: first 4 story files (~80 MB) + first 50 MB of Twitter data.

In [2]:
texts = load_corpus()

Loading story-data (first 4 files)...
  Loaded: story_data_1.txt  (18.8 MB)
  Loaded: story_data_10.txt  (23.1 MB)
  Loaded: story_data_11.txt  (16.3 MB)
  Loaded: story_data_12.txt  (26.2 MB)
Loading twitter data (first 50 MB)...
  Loaded: tweets.txt  (first 50 MB)
Total raw characters: 68,313,531


## 3. Preprocessing

Steps applied:
- Diacritic (tashkeel) removal
- Arabic letter normalization (`أ إ آ` → `ا`, `ة` → `ه`, etc.)
- Noise removal: URLs, mentions, hashtags
- Franco-Arabe digits preserved (3=ع, 7=ح, 9=ق)
- Sentence segmentation + `<s>` / `</s>` boundary tokens

In [3]:
sentences = preprocess(texts)
print(f"Total sentences : {len(sentences):,}")
print(f"Example sentence: {sentences[0]}")

Total sentences : 108,970
Example sentence: ['<s>', 'تبسماات', 'بتكااسل', 'قبلاات', 'خدوودو', 'و', 'قالت', 'سمح', 'ليااا', 'و', 'لكن', 'شحاال', 'مانعست', 'هكاا', 'زين', 'العابدين', 'امممم', 'اذا', 'انا', 'ابلي', 'جيدا', 'ضحكات', 'و', 'قالت', 'لهلا', 'يحرمني', 'منك', 'قبل', 'يدها', 'و', 'قال', 'و', 'لا', 'منك', 'نوضو', '</s>']


In [4]:
def split(sentences, train_ratio=0.8, val_ratio=0.1):
    random.shuffle(sentences)
    n = len(sentences)
    t = int(n * train_ratio)
    v = int(n * (train_ratio + val_ratio))
    return sentences[:t], sentences[t:v], sentences[v:]

train, val, test = split(sentences)
print(f"Train : {len(train):,}")
print(f"Val   : {len(val):,}")
print(f"Test  : {len(test):,}")

Train : 87,176
Val   : 10,897
Test  : 10,897


## 4. Training

Words appearing fewer than 5 times are replaced with `<UNK>` to reduce vocabulary sparsity.
Both a **bigram** and **trigram** model are trained for comparison.

In [5]:
print("Training bigram model...")
bigram_model = NgramModel(order=2)
bigram_model.train(train)

Training bigram model...
  Rare words (freq < 5): 507,847 replaced with <UNK>
  Counting n-grams...
  Vocabulary    : 108,765 types
  Tokens        : 9,010,068
  Bigram ctxs   : 108,764
  Estimating λ weights via deleted interpolation...
  λ = (unigram=0.435, bigram=0.565)


In [6]:
print("Training trigram model...")
trigram_model = NgramModel(order=3)
trigram_model.train(train)

Training trigram model...
  Rare words (freq < 5): 507,847 replaced with <UNK>
  Counting n-grams...
  Vocabulary    : 108,765 types
  Tokens        : 9,010,068
  Bigram ctxs   : 108,764
  Trigram ctxs  : 3,770,103
  Estimating λ weights via deleted interpolation...
  λ = (unigram=0.416, bigram=0.369, trigram=0.215)


## 5. Evaluation — Perplexity

$$PP = 2^{-\frac{1}{N} \sum \log_2 P(w_i \mid w_{i-2}, w_{i-1})}$$

Lower perplexity = better model fit.

In [7]:
results = {}
for name, model in [("Bigram", bigram_model), ("Trigram", trigram_model)]:
    results[name] = {
        "Train":      perplexity(model, train[:5_000]),
        "Validation": perplexity(model, val),
        "Test":       perplexity(model, test),
    }

print(f"{'Model':<10} {'Train':>12} {'Validation':>12} {'Test':>12}")
print("-" * 48)
for name, scores in results.items():
    print(f"{name:<10} {scores['Train']:>12,.2f} {scores['Validation']:>12,.2f} {scores['Test']:>12,.2f}")

Model             Train   Validation         Test
------------------------------------------------
Bigram           283.13     3,547.92     3,977.42
Trigram           19.79     3,129.09     3,490.19


**Interpretation:**
- The trigram model achieves lower perplexity than bigram on all splits, confirming the value of wider context.
- The train/validation gap is expected for word-level n-gram models on noisy social media text: the vocabulary is large relative to the corpus, so many contexts seen at test time were never observed during training. This is a known limitation of count-based models.

## 6. Text Generation

Two strategies:
- **Sampling**: draws the next word proportionally to its probability
- **Greedy (top-5)**: samples from the 5 highest-probability words to avoid deterministic loops

In [8]:
print("=== Sampling ===")
for i in range(5):
    print(f"  {i+1}. {generate(trigram_model, strategy='sample')}")

=== Sampling ===
  1. هدر بفمو و كتحس بكرشها مخرجه عينيها فبلاصتها شويه مع شافت وجهو رجع شاف فيها وهم بلاد بردو لا والله خسات وقل عينيه عاد فقت شفت اجلال ياا من عذاب
  2. البنات هاد العام كنزه احم خالتي نمشي
  3. عادي حيت كبر و رجع قال بعض لقات راسها عند مالين الدار كنت فنيويورك
  4. نتا هبط قرب يبكي وحتى تسمع شي اصوات لعبه elite البارح ذبح لغير الله من كنا او خرجت و مدیریت غلط اعتراض وهي اصلا هاد بدات كتشطح بشعرها عرياان جاب
  5. جود اوكي سفيان يه اولدي ماجاوبهاش بحال ولدو ودابا دخلنا حتى حد لفخض وصدرو عريان واجد


In [9]:
print("=== Greedy (top-5) ===")
for i in range(3):
    print(f"  {i+1}. {generate(trigram_model, strategy='greedy')}")

=== Greedy (top-5) ===
  1. وا غا اجي عاونيني ناضت خت صاحبو يعني
  2. واخه هي لوله ا بنتي و انا و هي و لا
  3. هوا اللي و هي هي انا


In [10]:
print("=== Seeded generation ===")
for seed in ["واش نتا", "كنت نحب", "الله يعطيك"]:
    print(f"  [{seed}] → {generate(trigram_model, seed=seed, strategy='sample')}")

=== Seeded generation ===
  [واش نتا] → واش نتا كطير وهيا ففراشهاا مجموعه شركات فالمانيا كنتي ديما ونضت ولا معاها شي العاب صغار مونيكات قدام عيني بالم حارق كان موجد عينيها غير كتحلم حلم الشي لي كانت
  [كنت نحب] → كنت نحب نعرف من نيتو ههه من شرودها بكلامو مراتي فيه بالنسبه للبقيه تنهد وشدها من عبد الله كذلك حاجه هي غي لون حمر بغا يبدا التحريض على القتل اللي
  [الله يعطيك] → الله يعطيك صحه بشكل عام وقلوبكم في سلامه


## 7. Save Model

In [11]:
trigram_model.save("ngram_model.pkl")
print("Trigram model saved.")

Model saved → ngram_model.pkl
Trigram model saved.
